# 03 — Feature Engineering
**Starbucks Customer Segmentation**

This notebook builds customer-level behavioral features from the event log for use in clustering.

---

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/cleaned_data.csv')
print('Data loaded. Shape:', df.shape)

## 2. Build Offer-Level Features per Customer

In [ ]:
# Separate events
received  = df[df['event'] == 'offer received']
viewed    = df[df['event'] == 'offer viewed']
completed = df[df['event'] == 'offer completed']
transact  = df[df['event'] == 'transaction']

# Count per customer
offers_received  = received.groupby('customer_id').size().rename('offers_received')
offers_viewed    = viewed.groupby('customer_id').size().rename('offers_viewed')
offers_completed = completed.groupby('customer_id').size().rename('offers_completed')
total_transactions = transact.groupby('customer_id').size().rename('total_transactions')
total_spent = transact.groupby('customer_id')['amount'].sum().rename('total_spent')

print('Per-customer offer counts computed.')

## 3. Build Customer-Level Feature Table

In [ ]:
# Base: one row per customer with demographics
customer_df = df.drop_duplicates('customer_id')[['customer_id','age','income','gender',
                                                  'membership_days']].copy()

# Merge behavioral features
customer_df = customer_df.set_index('customer_id')
customer_df = customer_df.join([offers_received, offers_viewed,
                                 offers_completed, total_transactions, total_spent])
customer_df.fillna(0, inplace=True)

print('Customer feature table shape:', customer_df.shape)
display(customer_df.head())

## 4. Derived Ratio Features

In [ ]:
# View rate: how often customer views an offer after receiving it
customer_df['view_rate'] = np.where(
    customer_df['offers_received'] > 0,
    customer_df['offers_viewed'] / customer_df['offers_received'], 0
)

# Completion rate: how often customer completes after receiving
customer_df['completion_rate'] = np.where(
    customer_df['offers_received'] > 0,
    customer_df['offers_completed'] / customer_df['offers_received'], 0
)

# Average spend per transaction
customer_df['avg_spend'] = np.where(
    customer_df['total_transactions'] > 0,
    customer_df['total_spent'] / customer_df['total_transactions'], 0
)

print('Derived features added.')
display(customer_df[['view_rate','completion_rate','avg_spend']].describe())

## 5. Encode Categorical Features

In [ ]:
# One-hot encode gender
customer_df = pd.get_dummies(customer_df, columns=['gender'], drop_first=True)

print('Final feature table shape:', customer_df.shape)
print('Columns:', customer_df.columns.tolist())

## 6. Save Feature Table

In [ ]:
customer_df.to_csv('../data/customer_features.csv')
print('Feature table saved to ../data/customer_features.csv')
print('Shape:', customer_df.shape)